# PatchCore VisA Training – Standard / FR / Masked / FR+Masked

Train and evaluate four PatchCore variants on all 12 VisA product categories.
Artefacts (memory banks, thresholds, calibrators, heatmaps) are packaged into
a downloadable ZIP at the end of the notebook.

## Prerequisites

1. **Dataset** – Attach the *VisA* dataset as a Kaggle input (dataset slug
   `visa` or `visa-dataset`).  The notebook auto-detects the root path.
2. **Accelerator** – Enable GPU (P100 or T4) in *Notebook settings → Accelerator*.
3. **Internet** – Required the first time to download the WideResNet-50-2
   ImageNet weights from `torchvision`.

> **Disk budget** – Each variant × category produces ~50 MB of artefacts.
> Running all 4 variants × 12 categories needs ~2–3 GB of free space.

In [ ]:
# Install / upgrade packages required by the pipeline.
# scikit-learn  – calibration & metrics
# scikit-image  – heatmap / contour drawing in visualization.py
# tqdm          – progress bars
# pyyaml        – config.yaml parsing
# joblib        – calibrator serialisation
!pip install -q scikit-learn scikit-image tqdm pyyaml joblib

In [ ]:
"""System setup: add the repo to sys.path and configure logging."""
import os
import sys
import logging
from pathlib import Path

# ── Locate the cloned repository ──────────────────────────────────────────
# Adjust REPO_ROOT if you cloned to a different location.
_CANDIDATE_ROOTS = [
    "/kaggle/working/patchcore-visa",
    "/kaggle/working/visa",
    str(Path.cwd()),           # current notebook directory
    str(Path.cwd().parent),    # one level up
]

REPO_ROOT = None
for _root in _CANDIDATE_ROOTS:
    if (Path(_root) / "src").is_dir():
        REPO_ROOT = _root
        break

if REPO_ROOT is None:
    raise RuntimeError(
        "Cannot locate the repository root (expected a 'src/' sub-directory).\n"
        "Clone the repo or adjust _CANDIDATE_ROOTS above."
    )

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Repository root: {REPO_ROOT}")

# ── Configure logging ──────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
    datefmt="%H:%M:%S",
)

# ── Core imports ────────────────────────────────────────────────────────────
import torch
import numpy as np
import yaml

from src.dataset import CATEGORIES, find_visa_root
from src.artifacts import VARIANTS

print(f"PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}")
print(f"VisA categories: {CATEGORIES}")

In [ ]:
"""Locate the VisA dataset and display a summary of available images."""
from pathlib import Path

# Search common Kaggle input paths as well as the working directory
KAGGLE_SEARCH_PATHS = [
    "/kaggle/input/visa/visa_pytorch/",
    "/kaggle/input/visa/",
    "/kaggle/input/visad/",
    "/kaggle/input/visa-dataset/",
    "/kaggle/input/visual-anomaly-visa/",
    str(Path.cwd()),
]

VISA_ROOT = find_visa_root(search_paths=KAGGLE_SEARCH_PATHS)

if VISA_ROOT is None:
    raise RuntimeError(
        "VisA dataset not found.\n"
        "Attach it as a Kaggle input dataset (slug: visa or visa-dataset) "
        "or set VISA_ROOT manually below."
    )

# Uncomment and edit the line below to override auto-detection:
# VISA_ROOT = "/kaggle/input/visa/"

print(f"VisA root: {VISA_ROOT}\n")

# Show per-category image counts
print(f"{'Category':<16}  {'Normal':>8}  {'Anomaly':>9}  {'Masks':>7}")
print("-" * 44)
for cat in CATEGORIES:
    cat_root = Path(VISA_ROOT) / cat / "Data"
    n_normal  = len(list((cat_root / "Images" / "Normal").glob("*")))  if (cat_root / "Images" / "Normal").exists()  else 0
    n_anomaly = len(list((cat_root / "Images" / "Anomaly").glob("*"))) if (cat_root / "Images" / "Anomaly").exists() else 0
    n_masks   = len(list((cat_root / "Masks"  / "Anomaly").glob("*"))) if (cat_root / "Masks"  / "Anomaly").exists() else 0
    print(f"{cat:<16}  {n_normal:>8}  {n_anomaly:>9}  {n_masks:>7}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# USER CONFIGURATION – edit this cell to customise the run
# ══════════════════════════════════════════════════════════════════════════

# Path to the YAML config file (relative to REPO_ROOT)
CONFIG_PATH = str(Path(REPO_ROOT) / "kaggle" / "config.yaml")

# Root directory for all exported artefacts
OUTPUT_DIR = "/kaggle/working/exports"

# Variants to train – subset of ['standard', 'fr', 'mask', 'fr_mask']
# Comment out variants you don't need to save time.
SELECTED_VARIANTS = [
    "standard",
    "fr",
    "mask",
    "fr_mask",
]

# Categories to process – subset of the 12 VisA categories.
# Set to None to process all categories from the config file.
SELECTED_CATEGORIES = None  # None → use all from config.yaml

# Device: 'auto' lets the pipeline pick GPU if available
DEVICE = "auto"

# Set to True to run only the first category as a quick smoke test
DRY_RUN = False

# ── Load and (optionally) override YAML config ─────────────────────────────
with open(CONFIG_PATH) as _fh:
    CONFIG = yaml.safe_load(_fh)

# Apply notebook overrides on top of the YAML values
CONFIG["export"]["output_dir"] = OUTPUT_DIR
if SELECTED_CATEGORIES is not None:
    CONFIG["dataset"]["categories"] = SELECTED_CATEGORIES

print("Active configuration:")
print(yaml.dump(CONFIG, default_flow_style=False))

In [ ]:
"""Run the full training + evaluation pipeline.

This cell imports the train_kaggle helper functions directly so that
output appears inline in the notebook.  Alternatively you can call the
script from the command line::

    !python {REPO_ROOT}/kaggle/train_kaggle.py --config {CONFIG_PATH}
"""
import time
from pathlib import Path
from tqdm.notebook import tqdm

# Import pipeline helpers from train_kaggle.py
sys.path.insert(0, str(Path(REPO_ROOT) / "kaggle"))
from train_kaggle import (
    resolve_device,
    run_variant,
)
from src.artifacts import save_metrics_summary, package_artifacts

# ── Resolve device ─────────────────────────────────────────────────────────
device = resolve_device(
    cfg_device=CONFIG["training"]["device"],
    cli_device=DEVICE,
)
print(f"Device: {device}")

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

categories = CONFIG["dataset"]["categories"]
if DRY_RUN:
    categories = categories[:1]
    print(f"DRY RUN – processing only: {categories}")

t0 = time.perf_counter()
ALL_RESULTS = {}

for variant in tqdm(SELECTED_VARIANTS, desc="Variants"):
    variant_results = run_variant(
        variant=variant,
        categories=categories,
        visa_root=VISA_ROOT,
        cfg=CONFIG,
        output_dir=output_dir,
        device=device,
        dry_run=DRY_RUN,
    )
    ALL_RESULTS[variant] = variant_results

# Save combined metrics summary
overall = {
    v: {cat: d["metrics"] for cat, d in cats.items()}
    for v, cats in ALL_RESULTS.items()
}
save_metrics_summary(overall, output_dir / "metrics_summary.json")

elapsed = time.perf_counter() - t0
print(f"\nTotal time: {elapsed/60:.1f} min")

In [ ]:
"""Display a summary metrics table using pandas."""
import json
import pandas as pd

metrics_path = output_dir / "metrics_summary.json"
with open(metrics_path) as fh:
    summary = json.load(fh)

rows = []
for variant, cats in summary.items():
    for cat, m in cats.items():
        rows.append({
            "variant":    variant,
            "category":   cat,
            "img_auroc":  round(m.get("image_auroc",  float("nan")), 4),
            "img_auprc":  round(m.get("image_auprc",  float("nan")), 4),
            "px_auroc":   round(m.get("pixel_auroc",  float("nan")), 4),
            "px_auprc":   round(m.get("pixel_auprc",  float("nan")), 4),
            "img_f1":     round(m.get("img_f1",       float("nan")), 4),
        })

df = pd.DataFrame(rows)

# Per-variant mean AUROC for a quick overview
mean_row = (
    df.groupby("variant")[["img_auroc", "px_auroc", "img_f1"]]
    .mean()
    .round(4)
)
print("Mean metrics per variant:")
print(mean_row.to_string())
print()

# Full per-category table
pd.set_option("display.max_rows", None)
df.style.background_gradient(subset=["img_auroc", "px_auroc"], cmap="RdYlGn")

In [ ]:
"""Display a grid of sample heatmap overlays for visual inspection."""
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Pick the first available variant and category to display
_display_variant  = SELECTED_VARIANTS[0]
_display_category = CONFIG["dataset"]["categories"][0]

# Search for overlay.png files saved during test scoring
_heatmap_dir = (
    output_dir / _display_variant / _display_category / "heatmaps" / "test"
)

_overlay_files = sorted(_heatmap_dir.rglob("overlay.png")) if _heatmap_dir.exists() else []

if not _overlay_files:
    print(
        f"No overlay images found in {_heatmap_dir}.\n"
        "Make sure training.save_heatmaps_test is True in config.yaml."
    )
else:
    n_show = min(8, len(_overlay_files))
    ncols = 4
    nrows = (n_show + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes.flatten()

    for ax, img_path in zip(axes, _overlay_files[:n_show]):
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(img_path.parent.name, fontsize=8)
        ax.axis("off")

    # Hide unused axes
    for ax in axes[n_show:]:
        ax.set_visible(False)

    fig.suptitle(
        f"Sample heatmaps – variant={_display_variant}  category={_display_category}",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()
    print(f"Showing {n_show} of {len(_overlay_files)} saved overlays.")

In [ ]:
"""Package all artefacts into a ZIP file and show download instructions."""
from src.artifacts import package_artifacts

zip_name = CONFIG["export"]["zip_name"]
zip_path = output_dir / zip_name

package_artifacts(
    export_root=output_dir,
    output_zip_path=zip_path,
)

zip_mb = zip_path.stat().st_size / 1e6
print(f"ZIP created: {zip_path}  ({zip_mb:.1f} MB)")
print()
print("To download from Kaggle:")
print("  1. Open the notebook output panel (right sidebar).")
print(f"  2. Find '{zip_name}' and click the download icon.")
print()
print("Or copy to /kaggle/working/ for the Output Files panel:")
print(f"  import shutil")
print(f"  shutil.copy('{zip_path}', '/kaggle/working/{zip_name}')")

## Next Steps

### Download and use artefacts
1. **Download the ZIP** from the Kaggle output panel (see cell above).
2. **Extract** the ZIP; the directory structure is:
   ```
   exports/
   ├── standard/
   │   ├── thresholds.json
   │   ├── calibrators/<category>.pkl
   │   └── <category>/
   │       ├── memory_bank.npz
   │       └── config.json
   ├── fr/  mask/  fr_mask/   (same layout)
   └── metrics_summary.json
   ```

### Run inference locally
Load a fitted model with:
```python
from src.artifacts import load_model_artifacts
artefacts = load_model_artifacts(
    model_dir="/path/to/exports",
    variant="standard",
    category="candle",
    device="cpu",
)
model      = artefacts["model"]
thresholds = artefacts["thresholds"]
calibrator = artefacts["calibrator"]
```

### Launch the GUI on Windows
After copying the artefacts to your Windows machine, start the
interactive inspection GUI with:
```
python gui/app.py --model-dir exports --variant standard
```

### Improve results
- Increase `coreset_ratio` (e.g. 0.25) to keep more memory-bank patches.
- Tune `fr_pca_components` for the FR variants.
- Use `DRY_RUN = True` in cell 6 to iterate quickly on a single category.